In [1]:
from ingest import load_faq_data, chunk_faq_data, embed_texts, ensure_faq_table_exists, get_db_connection, insert_faq_chunks
from embedder import Embedder
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openai_client = OpenAI()

In [3]:
documents = load_faq_data()
chunks = chunk_faq_data(documents, chunk_size=500, overlap=100)
embeddings = embed_texts([chunk['content'] for chunk in chunks])

  0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
ensure_faq_table_exists()
insert_faq_chunks(chunks, embeddings)

In [5]:
conn = get_db_connection()
embedder = Embedder()
vector_assistant = RAGBase(
    embedder=embedder,
    conn=conn,
    llm_client=openai_client,
)

In [6]:
results = vector_assistant.rag("How do I start saving for retirement?", num_results=5)

Context for query 'How do I start saving for retirement?':
Title: Épargne-pension
Language: fr
Content: Question: Épargne-pension
Answer: - Ta pension est encore loin, mais plus tu commences tôt à épargner pour ta pension, plus tu en tireras des avantages.

- C'est fiscalement avantageux.

- Cela te permettra de disposer d'une réserve supplémentaire pour tes vieux jours.

 Tout savoir sur l'épargne-pension

Title: Ton premier salaire
Language: fr
Content: Question: Ton premier salaire
Answer: - Vérifie ta fiche de paie dès que tu la reçois.

- Vérifie ton salaire brut, les montants qui sont retenus et ce qu'il te reste comme salaire net.

- Conseil : mets chaque mois une partie de ton salaire de côté comme réserve d'argent (épargne). Tu peux le faire automatiquement via un ordre permanent.

 Brut, net : quelle est la différence ?

Title: Wat is de impact van je gezinssituatie op je pensioen?
Language: nl
Content: rkt. Vul daarvoor het formulier in, laat het ondertekenen door je ziekenf

In [7]:
results

'According to the context, you can start saving for retirement by beginning as early as possible. It says the earlier you start, the more benefit you get, and that retirement saving is tax-friendly and can give you extra money for later in life.'

In [8]:
conn.close()